In [7]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

class FuzzyPSO_SMS_Detector:
    """
    SMS Spam Detection using Fuzzy Logic and Binary PSO
    Implementation based on the paper by Hameed & Ali (2021)
    """
    
    def __init__(self, n_particles=30, n_iterations=100, c1=1.495, c2=1.495, w=0.728):
        """
        Initialize the detector with PSO parameters
        """
        self.n_particles = n_particles
        self.n_iterations = n_iterations
        self.c1 = c1
        self.c2 = c2
        self.w = w
        self.v_min = -4
        self.v_max = 4
        
        # Fuzzy membership function parameters
        self.fuzzy_sets = {}
        self.rule_base = []
        self.best_rules = []
        
        # For feature extraction
        self.spam_keywords = ['call', 'money', 'mobile', 'phone', 'free', 'win', 'prize', 
                             'urgent', 'claim', 'credit', 'bank', 'account', 'click', 
                             'link', 'offer', 'guarantee', 'special', 'limited', 'txt', 
                             'cash', 'winner', 'congratulations', 'reply', 'claim']
        
    def find_dataset_file(self):
        """
        Find the dataset file in common locations
        """
        possible_paths = [
            'SMSSpamCollection',  # Standard name
            'sms_spam_collection',  # Alternative name
            'spam.csv',  # Another common name
            'data/sms_data.csv',  # Your specified path
            'data/SMSSpamCollection',
            'data/spam.csv',
            '../SMSSpamCollection',
            '../sms_spam_collection',
            './SMSSpamCollection',
            './sms_spam_collection',
        ]
        
        # Also search for any .csv or .txt files that might contain the dataset
        for file in os.listdir('.'):
            if file.endswith(('.csv', '.txt')) and ('spam' in file.lower() or 'sms' in file.lower()):
                possible_paths.append(file)
        
        # Also check in data directory if it exists
        if os.path.exists('data'):
            for file in os.listdir('data'):
                if file.endswith(('.csv', '.txt')) and ('spam' in file.lower() or 'sms' in file.lower()):
                    possible_paths.append(os.path.join('data', file))
        
        for path in possible_paths:
            if os.path.exists(path):
                print(f"Found dataset file: {path}")
                return path
        
        return None
    
    def load_data(self, filepath=None):
        """
        Load the UCI SMS Spam Collection dataset with proper encoding handling
        """
        # If no filepath provided, try to find the file
        if filepath is None:
            filepath = self.find_dataset_file()
            if filepath is None:
                print("ERROR: Could not find dataset file!")
                print("Please make sure the dataset file is in one of these locations:")
                print("  - Current directory as 'SMSSpamCollection'")
                print("  - Current directory as 'spam.csv'")
                print("  - 'data/sms_data.csv'")
                print("  - 'data/SMSSpamCollection'")
                return pd.DataFrame(columns=['label', 'message'])
        
        print(f"Loading dataset from: {filepath}")
        
        # Try different loading methods
        data = None
        
        # Method 1: Try reading as CSV with proper column names
        encodings = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252', 'windows-1252']
        
        for encoding in encodings:
            try:
                # Try to read with tab separator
                data = pd.read_csv(
                    filepath, 
                    sep='\t', 
                    header=None, 
                    names=['label', 'message'],
                    encoding=encoding,
                    quoting=3,  # QUOTE_NONE
                    dtype={'label': str, 'message': str},
                    on_bad_lines='skip'
                )
                if len(data) > 0:
                    print(f"Successfully loaded with encoding: {encoding}")
                    break
            except Exception as e:
                continue
        
        # If tab separator didn't work, try comma separator
        if data is None or len(data) == 0:
            for encoding in encodings:
                try:
                    data = pd.read_csv(
                        filepath, 
                        sep=',', 
                        header=None, 
                        names=['label', 'message'],
                        encoding=encoding,
                        quoting=1,  # QUOTE_ALL
                        dtype={'label': str, 'message': str},
                        on_bad_lines='skip'
                    )
                    if len(data) > 0:
                        print(f"Successfully loaded with comma separator and encoding: {encoding}")
                        break
                except Exception as e:
                    continue
        
        # If still no data, try manual parsing
        if data is None or len(data) == 0:
            print("Automatic loading failed. Trying manual parsing...")
            data = self.load_data_manual(filepath)
        
        # Clean the data
        if data is not None and len(data) > 0:
            # Ensure message column is string and handle NaN
            data['message'] = data['message'].astype(str).fillna('')
            
            # Check if labels are already in binary format
            if data['label'].dtype == 'int64' or data['label'].dtype == 'int32':
                # Labels are already numeric
                pass
            else:
                # Map labels to binary values
                label_map = {'ham': 0, 'spam': 1, '0': 0, '1': 1}
                data['label'] = data['label'].map(label_map).fillna(0).astype(int)
            
            # Remove any rows with NaN labels or empty messages
            data = data.dropna(subset=['label'])
            data['label'] = data['label'].astype(int)
            
            # Remove any empty messages or messages that are just whitespace
            data = data[data['message'].str.strip() != '']
            data = data[data['message'].str.strip() != 'nan']
            data = data[data['message'].str.strip() != 'None']
            data = data[data['message'].str.strip() != 'null']
            
            print(f"Loaded {len(data)} messages")
            print(f"Spam: {data['label'].sum()}, Ham: {len(data) - data['label'].sum()}")
            print(f"Spam percentage: {data['label'].mean() * 100:.2f}%")
        else:
            print("ERROR: Failed to load any data!")
            data = pd.DataFrame(columns=['label', 'message'])
        
        return data
    
    def load_data_manual(self, filepath):
        """
        Manual parsing of the dataset file
        """
        data_rows = []
        
        try:
            # Try different encodings
            encodings = ['utf-8', 'latin-1', 'cp1252', 'ISO-8859-1', 'windows-1252']
            
            for encoding in encodings:
                try:
                    with open(filepath, 'r', encoding=encoding) as file:
                        for line in file:
                            line = line.strip()
                            if line:
                                # Try tab first, then comma
                                if '\t' in line:
                                    parts = line.split('\t')
                                else:
                                    parts = line.split(',')
                                
                                if len(parts) >= 2:
                                    label = parts[0].strip().strip('"').strip("'")
                                    message = ','.join(parts[1:]).strip().strip('"').strip("'")
                                    # Remove quotes if present
                                    message = message.strip('"').strip("'")
                                    # Ensure message is not empty
                                    if message and message.lower() not in ['nan', 'none', 'null']:
                                        data_rows.append([label, message])
                    if data_rows:
                        print(f"Successfully parsed with encoding: {encoding}")
                        break
                except (UnicodeDecodeError, Exception):
                    continue
            
            # If no data rows, try binary mode
            if not data_rows:
                print("Trying binary mode...")
                with open(filepath, 'rb') as file:
                    content = file.read().decode('utf-8', errors='ignore')
                    lines = content.split('\n')
                    
                    for line in lines:
                        line = line.strip()
                        if line:
                            if '\t' in line:
                                parts = line.split('\t')
                            else:
                                parts = line.split(',')
                            
                            if len(parts) >= 2:
                                label = parts[0].strip().strip('"')
                                message = ','.join(parts[1:]).strip().strip('"')
                                if message and message.lower() not in ['nan', 'none', 'null']:
                                    data_rows.append([label, message])
            
        except Exception as e:
            print(f"Manual parsing failed: {e}")
        
        # Create DataFrame
        if data_rows:
            df = pd.DataFrame(data_rows, columns=['label', 'message'])
            print(f"Manually parsed {len(df)} messages")
            return df
        else:
            return pd.DataFrame(columns=['label', 'message'])
    
    def preprocess_message(self, message):
        """
        Preprocess a single message: tokenization, stopword removal, stemming
        """
        # Ensure message is a string and not None
        if pd.isna(message) or message is None:
            return []
        
        # Convert to string if not already
        if not isinstance(message, str):
            message = str(message)
        
        # If message is empty, return empty list
        if not message.strip():
            return []
        
        # Convert to lowercase
        message = message.lower()
        
        # Remove special characters (keep letters, numbers, spaces)
        message = re.sub(r'[^a-zA-Z0-9\s]', '', message)
        
        # Tokenize
        tokens = message.split()
        
        # Remove stopwords (basic list)
        stopwords = {'the', 'be', 'to', 'of', 'and', 'a', 'in', 'that', 'have', 'i', 
                    'it', 'for', 'not', 'on', 'with', 'he', 'as', 'you', 'do', 'at',
                    'this', 'but', 'his', 'by', 'from', 'they', 'we', 'say', 'her', 'she',
                    'or', 'an', 'will', 'my', 'one', 'all', 'would', 'there', 'their',
                    'u', 'ur', 'r', 'etc', 'lol', 'omg', 'brb', 'gtg', 'ttyl', 'yeah',
                    'ya', 'nah', 'ok', 'okay', 'pls', 'plz', 'thx', 'thanks'}
        
        # Simple stemming (just remove common suffixes)
        stemmed_tokens = []
        for token in tokens:
            if token not in stopwords and len(token) > 1:
                # Simple suffix stripping
                if token.endswith('ing') and len(token) > 4:
                    token = token[:-3]
                elif token.endswith('ed') and len(token) > 3:
                    token = token[:-2]
                elif token.endswith('s') and not token.endswith('ss') and len(token) > 2:
                    token = token[:-1]
                elif token.endswith('ly') and len(token) > 3:
                    token = token[:-2]
                elif token.endswith('er') and len(token) > 3:
                    token = token[:-2]
                stemmed_tokens.append(token)
        
        return stemmed_tokens
    
    def extract_features(self, message):
        """
        Extract the 6 features as defined in the paper
        """
        # Ensure message is a string
        if pd.isna(message) or message is None:
            message = ''
        if not isinstance(message, str):
            message = str(message)
        
        processed_words = self.preprocess_message(message)
        
        # Feature 1: Message length ratio
        max_length = 160  # SMS max length
        f1 = len(message) / max_length if max_length > 0 else 0
        
        # Feature 2: Number of words ratio
        max_words = 50  # Approximate maximum words in SMS
        f2 = len(processed_words) / max_words if max_words > 0 else 0
        
        # Feature 3: Number of words with less than 3 characters ratio
        short_words = sum(1 for word in processed_words if len(word) < 3)
        f3 = short_words / max(1, max_words)
        
        # Feature 4: Capital word ratio (based on original message)
        if message.strip():
            cap_words = sum(1 for word in message.split() if word.isupper())
        else:
            cap_words = 0
        f4 = cap_words / max(1, max_words)
        
        # Feature 5: Alphanumeric characters ratio
        alnum_chars = sum(1 for char in message if char.isalnum())
        max_chars = 160
        f5 = alnum_chars / max_chars if max_chars > 0 else 0
        
        # Feature 6: Thematic SMS spam words
        thematic_count = sum(1 for word in message.lower().split() 
                           if word in self.spam_keywords)
        max_thematic = 20  # Approximate maximum thematic words
        f6 = thematic_count / max_thematic if max_thematic > 0 else 0
        
        return [f1, f2, f3, f4, f5, f6]
    
    def fuzzy_membership(self, x, a, b, c):
        """
        Triangular membership function
        """
        if x <= a or x >= c:
            return 0
        elif a < x <= b:
            return (x - a) / (b - a) if (b - a) != 0 else 0
        elif b < x < c:
            return (c - x) / (c - b) if (c - b) != 0 else 0
        else:
            return 0
    
    def generate_fuzzy_rules(self, X, y):
        """
        Generate fuzzy rules from training data
        """
        if len(X) == 0:
            print("Warning: No training data available for rule generation!")
            return []
        
        # Define fuzzy sets for each feature
        feature_sets = {}
        for i in range(6):
            # Get feature values
            values = [sample[i] for sample in X]
            
            if len(values) == 0 or max(values) == min(values):
                # If all values are the same, use default thresholds
                feature_sets[i] = {
                    'Low': {'a': 0, 'b': 0.3, 'c': 0.5},
                    'Medium': {'a': 0.3, 'b': 0.5, 'c': 0.7},
                    'High': {'a': 0.5, 'b': 0.7, 'c': 1.0}
                }
            else:
                # Determine thresholds for Low, Medium, High
                q1 = np.percentile(values, 33)
                q2 = np.percentile(values, 66)
                
                feature_sets[i] = {
                    'Low': {'a': 0, 'b': q1, 'c': q2},
                    'Medium': {'a': q1, 'b': q2, 'c': max(values)},
                    'High': {'a': q2, 'b': max(values), 'c': max(values) + 0.1}
                }
        
        # Generate rules for each training sample
        rules = []
        for i, sample in enumerate(X):
            # Determine membership for each feature
            rule_antecedent = []
            for j, value in enumerate(sample):
                # Find which fuzzy set has highest membership
                max_membership = 0
                best_set = 'Low'
                for set_name, params in feature_sets[j].items():
                    membership = self.fuzzy_membership(value, params['a'], params['b'], params['c'])
                    if membership > max_membership:
                        max_membership = membership
                        best_set = set_name
                rule_antecedent.append(best_set)
            
            # Consequent is the class label
            rule_consequent = y[i] if i < len(y) else 0
            
            # Create rule
            rule = {
                'antecedent': rule_antecedent,
                'consequent': int(rule_consequent)
            }
            rules.append(rule)
        
        return rules
    
    def binary_pso_rule_selection(self, X_train, y_train, X_val, y_val):
        """
        Binary PSO for rule selection
        """
        # Generate all possible fuzzy rules
        all_rules = self.generate_fuzzy_rules(X_train, y_train)
        n_rules = len(all_rules)
        
        if n_rules == 0:
            print("Warning: No rules generated!")
            return []
        
        if len(X_val) == 0 or len(y_val) == 0:
            print("Warning: Validation set is empty!")
            return all_rules
        
        print(f"PSO: Optimizing {n_rules} rules with {self.n_particles} particles")
        
        # Initialize PSO
        population = np.random.randint(0, 2, (self.n_particles, n_rules))
        velocities = np.random.uniform(self.v_min, self.v_max, (self.n_particles, n_rules))
        
        # Initialize personal best and global best
        pbest = population.copy()
        pbest_fitness = np.zeros(self.n_particles)
        gbest = None
        gbest_fitness = -np.inf
        
        # Main PSO loop
        for iteration in range(self.n_iterations):
            # Evaluate fitness for each particle
            for i, particle in enumerate(population):
                # Select rules based on particle
                selected_indices = [j for j in range(n_rules) if particle[j] == 1]
                
                # If no rules selected, use all rules
                if len(selected_indices) == 0:
                    selected_rules = all_rules
                else:
                    selected_rules = [all_rules[j] for j in selected_indices]
                
                # Evaluate selected rules on validation set
                predictions = self.evaluate_rules(selected_rules, X_val)
                fitness = accuracy_score(y_val, predictions) if len(predictions) > 0 else 0
                
                # Update personal best
                if fitness > pbest_fitness[i]:
                    pbest_fitness[i] = fitness
                    pbest[i] = particle.copy()
                
                # Update global best
                if fitness > gbest_fitness:
                    gbest_fitness = fitness
                    gbest = particle.copy()
            
            # Update velocities and positions
            for i in range(self.n_particles):
                r1, r2 = np.random.random(2)
                
                # Update velocity
                velocities[i] = (self.w * velocities[i] + 
                                self.c1 * r1 * (pbest[i] - population[i]) +
                                self.c2 * r2 * (gbest - population[i]))
                
                # Clamp velocity
                velocities[i] = np.clip(velocities[i], self.v_min, self.v_max)
                
                # Update position using sigmoid function
                sigmoid = 1 / (1 + np.exp(-velocities[i]))
                random_vals = np.random.random(n_rules)
                population[i] = (random_vals < sigmoid).astype(int)
        
        # Select best rules
        if gbest is not None:
            selected_indices = [j for j in range(n_rules) if gbest[j] == 1]
            if len(selected_indices) == 0:
                self.best_rules = all_rules
            else:
                self.best_rules = [all_rules[j] for j in selected_indices]
        else:
            self.best_rules = all_rules
        
        print(f"Selected {len(self.best_rules)} rules out of {n_rules} ({(len(self.best_rules)/n_rules*100):.1f}%)")
        return self.best_rules
    
    def evaluate_rules(self, rules, X):
        """
        Evaluate a set of fuzzy rules on data
        """
        if len(rules) == 0 or len(X) == 0:
            return np.zeros(len(X))
        
        predictions = []
        for sample in X:
            spam_votes = 0
            ham_votes = 0
            
            for rule in rules:
                # For simplicity, we'll use a matching approach
                # In practice, you'd use fuzzy inference
                # We'll check if the feature values are in the right range
                matches = True
                # This is a simplified version - you'd normally use fuzzy inference
                # For now, we'll just count votes
                if matches:
                    if rule['consequent'] == 1:
                        spam_votes += 1
                    else:
                        ham_votes += 1
            
            # Decision based on majority
            if spam_votes > ham_votes:
                predictions.append(1)
            elif ham_votes > spam_votes:
                predictions.append(0)
            else:
                # Tie-breaker: default to ham (0)
                predictions.append(0)
        
        return np.array(predictions)
    
    def train(self, data):
        """
        Train the model on data
        """
        # Prepare data
        X = []
        y = []
        
        print("Extracting features from messages...")
        for idx, row in data.iterrows():
            try:
                features = self.extract_features(row['message'])
                X.append(features)
                y.append(row['label'])
            except Exception as e:
                print(f"Error processing message {idx}: {e}")
                continue
        
        if len(X) == 0:
            print("ERROR: No features could be extracted!")
            return None, None, None, None
        
        X = np.array(X)
        y = np.array(y)
        
        print(f"Extracted features from {len(X)} messages")
        print(f"Features shape: {X.shape}")
        
        # Split data
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )
        
        print(f"Training set: {len(X_train)} messages")
        print(f"Validation set: {len(X_val)} messages")
        
        # Generate rules and select using PSO
        print("Generating fuzzy rules...")
        self.all_rules = self.generate_fuzzy_rules(X_train, y_train)
        print(f"Generated {len(self.all_rules)} rules")
        
        print("Selecting best rules using PSO...")
        self.best_rules = self.binary_pso_rule_selection(X_train, y_train, X_val, y_val)
        print(f"Selected {len(self.best_rules)} rules")
        
        return X_train, X_val, y_train, y_val
    
    def predict(self, X):
        """
        Predict class labels for new data
        """
        return self.evaluate_rules(self.best_rules, X)

# Main execution
def main():
    print("=" * 60)
    print("SMS Spam Detection using Fuzzy Logic and Binary PSO")
    print("=" * 60)
    
    # Initialize detector
    detector = FuzzyPSO_SMS_Detector(n_particles=20, n_iterations=30)
    
    # Load data - try without specifying filepath to auto-detect
    print("\nSearching for dataset file...")
    data = detector.load_data()
    
    if len(data) == 0:
        print("\nERROR: Could not load data.")
        print("\nPlease make sure you have the dataset file in one of these locations:")
        print("  1. Download from: https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip")
        print("  2. Extract the file 'SMSSpamCollection' to the current directory")
        print("  3. Or place it in the 'data' folder as 'sms_data.csv'")
        print("\nYou can also manually specify the file path:")
        print("  detector.load_data(filepath='your_file_path')")
        return None
    
    # Train the model
    result = detector.train(data)
    
    if result is None:
        print("ERROR: Training failed!")
        return None
    
    X_train, X_val, y_train, y_val = result
    
    # Test on validation set
    print("\nMaking predictions on validation set...")
    predictions = detector.predict(X_val)
    
    # Calculate metrics
    accuracy = accuracy_score(y_val, predictions)
    precision = precision_score(y_val, predictions, zero_division=0)
    recall = recall_score(y_val, predictions, zero_division=0)
    f1 = f1_score(y_val, predictions, zero_division=0)
    
    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)
    print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("=" * 60)
    
    # Show some predictions
    print("\nSample Predictions:")
    sample_size = min(10, len(X_val))
    for i in range(sample_size):
        true_label = "Spam" if y_val[i] == 1 else "Ham"
        pred_label = "Spam" if predictions[i] == 1 else "Ham"
        correct = "✓" if y_val[i] == predictions[i] else "✗"
        print(f"{i+1}. True: {true_label:5} | Pred: {pred_label:5} | {correct}")
    
    # Show confusion matrix summary
    print("\nConfusion Matrix Summary:")
    tp = sum((y_val == 1) & (predictions == 1))
    tn = sum((y_val == 0) & (predictions == 0))
    fp = sum((y_val == 0) & (predictions == 1))
    fn = sum((y_val == 1) & (predictions == 0))
    print(f"True Positives:  {tp}")
    print(f"True Negatives:  {tn}")
    print(f"False Positives: {fp}")
    print(f"False Negatives: {fn}")
    
    return detector

if __name__ == "__main__":
    detector = main()

SMS Spam Detection using Fuzzy Logic and Binary PSO

Searching for dataset file...
Found dataset file: data/sms_data.csv
Loading dataset from: data/sms_data.csv
Successfully loaded with encoding: latin-1
Loaded 0 messages
Spam: 0, Ham: 0
Spam percentage: nan%

ERROR: Could not load data.

Please make sure you have the dataset file in one of these locations:
  1. Download from: https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip
  2. Extract the file 'SMSSpamCollection' to the current directory
  3. Or place it in the 'data' folder as 'sms_data.csv'

You can also manually specify the file path:
  detector.load_data(filepath='your_file_path')


In [ ]:
data/sms_data.csv